kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import anndata as ad
import pandas as pd
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

SERIES = "velmeshev"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

adata = ad.read_h5ad(os.path.join(clustered_h5ad_dir, f"{SERIES}.h5ad"))

In [ ]:
adata.obs["series"].value_counts()

project_id
PRJNA707311    171205
Name: count, dtype: int64

In [7]:
adata[adata.obs["leiden"] == "73"].obs["leiden"].value_counts()

leiden
73    2677
Name: count, dtype: int64

In [3]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "FERMT1", "CACNG5", "COL9A1", ],
    "COP": ["GPR17", "BMP4", "ENPP6", "BCAS1", "RGS16"],
    "Oligodendrocyte": ["MAG", "CNP", "MOBP", "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", "GRIN1", "GRIN2B", "MYT1L", ],
    "Excitatory": ["SLC17A7", "NEUROD2", "SATB2", "TBR1", "RORB", ],
    "Inhibitory": ["GAD1", "GAD2", "SLC32A1", "SLC6A1", ],
    "Microglia": ["P2RY12", "TMEM119", "SLC2A5", "SELPLG", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", ],
    "Endothelial": ["CLDN5", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "DCN", ],
    "RBC": ["HBB", "HBA1", "HBA2", ],
    "LowQuality": ["MALAT1", "NEAT1", ]
}

plot.plot_dotplot(adata, SERIES, cell_marker_genes, group = "leiden")

In [8]:
leiden_to_celltype = {
    "0": "?",
    "1": "OPC",
    "2": "?",
    "3": "?",
    "4": "?",
    "5": "?",
    "6": "?",
    "7": "?",
    "8": "?",
    "9": "?",
    "10": "?",
    "11": "?",
    "12": "?",
    "13": "?",
    "14": "?",
    "15": "?",
    "16": "?",
    "17": "?",
    "18": "?",
    "19": "?",
    "20": "?",
    "21": "?",
    "22": "?",
    "23": "?",
    "24": "?",
    "25": "?",
    "26": "?",
    "27": "?",
    "28": "?",
    "29": "?",
    "30": "?",
    "31": "?",
    "32": "?",
    "33": "?",
    "34": "?",
    "35": "?",
    "36": "?",
    "37": "?",
    "38": "?",
    "39": "?",
    "40": "?",
    "41": "?",
    "42": "?",
    "43": "?",
    "44": "?",
    "45": "?",
    "46": "?",
    "47": "?",
    "48": "?",
    "49": "?",
    "50": "?",
    "51": "?",
    "52": "?",
    "53": "?",
    "54": "?",
    "55": "?",
    "56": "?",
    "57": "?",
    "58": "?",
    "59": "?",
    "60": "?",
    "61": "?",
    "62": "?",
    "63": "?", 
    "64": "?",
    "65": "?",
    "66": "?",
    "67": "?",
    "68": "?",
    "69": "?",
    "70": "?",
    "71": "?",
    "72": "?",
    "73": "COP",
    "74": "?",
    "75": "Oligodendrocyte",
    "76": "?",
    "77": "?",
    "78": "?",
    "79": "OPC",
    "80": "?",
    "81": "?",
    "82": "?",
    "83": "?",
}

leiden_str = adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
adata.obs["cell"] = mapped
adata = adata[(adata.obs["cell"] != "Doublet") | (adata.obs["cell"] != "LowQuality")].copy()

In [73]:
adata.obs["celltype"] = adata.obs["cell"].copy() #type: ignore
plot.plot_umap(adata, SERIES, has_celltype=True)

In [9]:
oligo = adata[
    (adata.obs["cell"] == "OPC") |
    (adata.obs["cell"] == "COP") | 
    (adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))

/tmp/ipykernel_1832418/2854210309.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  oligo.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_oligo_raw.h5ad"))
